# 02-Bias-Analysis

After auditing the data for quality issues the next steps is to detect any bias patterns when assigning credit scores based on the EU AI Act. Credit scoring systems are classified as high-risk AI under EU AI Act Annex III. The following obligations have to be considered:

* Art. 10 (data governance)
* Art. 29 (human oversight)
* Art. 71 (penalties up to €30M or 6% global turnover for violations)

The following anaylsis implements mandatory bias monitoring, disparate impact analysis,
and fairness metrics required for high-risk AI systems operating in the EU.

In addition the EEOC Four-Fifths (80%) rule will be applied to identified potential bias categories. Under the AI Act (Art. 10 §5) and classical US fair-lending doctrine, a group whose approval rate falls below 80% of the highest group's rate is considered to suffer a disparate impact, triggering mandatory investigation.


## Setup and Connection

In [28]:
import json
import re
import pandas as pd
from datetime import date
from pathlib import Path
from pymongo import MongoClient


In [2]:
# Connect to local MongoDB
client = MongoClient('localhost', 27017)

# Create/access database and collection
db = client['project']
collection = db['credit_applications']
print("Connected to MongoDB successfully!")

Connected to MongoDB successfully!


In [3]:
# Load the json file
current_dir = Path.cwd()
repo_root = current_dir.parent
json_path = repo_root / "data" / "clean_credit_applications.json"

with open(json_path, 'r') as file:
    data = json.load(file)

print(f"Successfully loaded {len(data)} records from {json_path.name}")

Successfully loaded 499 records from clean_credit_applications.json


## Helper Function

Shared utilities used across all bias sections to ensure code consistency.

In [23]:
# Ordered age bands used in age-bias and intersectional analysis.
AGE_BUCKET_ORDER = ["<18", "18-24", "25-34", "35-44", "45-54", "55-64", "65+", "Unknown"]

def parse_dob(dob_raw):
    """Attempt to parse a date-of-birth string into a date object.

    Handles multiple formats found in the NovaCred dataset
    (e.g. '1990-05-21', '05/21/1990', '21.05.1990').

    Parameters
    ----------
    dob_raw : str | None
        The raw date-of-birth value from the document.

    Returns
    -------
    datetime.date | None
        Parsed date, or None if parsing fails.
    """
    if not dob_raw or not isinstance(dob_raw, str):
        return None

    # Normalise separators to '-'
    normalised = re.sub(r"[./]", "-", dob_raw.strip())
    parts = normalised.split("-")

    if len(parts) != 3:
        return None

    # Detect YYYY-MM-DD vs DD-MM-YYYY vs MM-DD-YYYY
    if len(parts[0]) == 4:          # YYYY-MM-DD
        year, month, day = parts
    elif len(parts[2]) == 4:        # DD-MM-YYYY or MM-DD-YYYY
        # Assume DD-MM-YYYY (European style common in EU datasets)
        day, month, year = parts
    else:
        return None

    try:
        return date(int(year), int(month), int(day))
    except (ValueError, TypeError):
        return None


def compute_age(dob_date, reference_date=None):
    """Compute age in whole years from a date object.

    Parameters
    ----------
    dob_date : datetime.date | None
        The parsed date of birth.
    reference_date : datetime.date | None
        Date to compute age against; defaults to today.

    Returns
    -------
    int | None
        Age in years, or None if dob_date is None or in the future.
    """
    if dob_date is None:
        return None

    ref = reference_date or date.today()

    if dob_date > ref:
        return None  # Invalid: future date of birth

    age = ref.year - dob_date.year
    # Subtract 1 if birthday hasn't occurred yet this year
    if (ref.month, ref.day) < (dob_date.month, dob_date.day):
        age -= 1

    return age


def get_age_band(dob_raw):
    """Map a raw date-of-birth string to a string age-band label.

    Parses the date string, computes current age, then buckets into
    the bands defined in AGE_BUCKET_ORDER.

    Parameters
    ----------
    dob_raw : str | None
        The applicant's raw date_of_birth value from MongoDB, or None.

    Returns
    -------
    str
        One of the labels defined in AGE_BUCKET_ORDER:
        'Unknown', '<18', '18-24', '25-34', '35-44',
        '45-54', '55-64', '65+'.
    """
    dob_date = parse_dob(dob_raw)
    age = compute_age(dob_date)

    if age is None:
        return "Unknown"
    if age < 18:
        return "<18"
    if age < 25:
        return "18-24"
    if age < 35:
        return "25-34"
    if age < 45:
        return "35-44"
    if age < 55:
        return "45-54"
    if age < 65:
        return "55-64"
    return "65+"


def avg_or_none(values):
    """Return the mean of *values*, or None if the list is empty."""
    return sum(values) / len(values) if values else None


def disparate_impact_report(results, label_width=20):
    """Apply the EEOC Four-Fifths (80%) rule and print a compliance table.

    Computes the disparate-impact ratio for every group relative to the group
    with the highest approval rate, flags groups whose ratio falls below 0.80,
    and returns a summary dict for use in the final compliance report.

    Parameters
    ----------
    results : list[dict]
        Each dict must have keys '_id' (group label), 'approval_rate' (float).
    label_width : int, optional
        Column width for the group-label column (default 20).

    Returns
    -------
    dict
        Mapping group label -> {'rate': float, 'ratio': float, 'pass': bool}.
        Returns an empty dict when fewer than two groups are present.
    """
    print("\n  Disparate Impact Analysis (80% Rule):")

    if len(results) < 2:
        print("  Insufficient groups for disparate-impact analysis.")
        return {}

    rates = {r["_id"]: r["approval_rate"] for r in results}
    max_rate = max(rates.values())
    max_group = max(rates, key=rates.get)

    print(f"\n  Reference group (highest rate): {max_group} → {max_rate * 100:.1f}%")
    print(f"  80% threshold: {max_rate * 0.8 * 100:.1f}%\n")

    summary = {}
    for r in sorted(results, key=lambda x: -x["approval_rate"]):
        group = r["_id"]
        rate = r["approval_rate"]
        ratio = rate / max_rate if max_rate > 0 else 0
        passed = ratio >= 0.8
        status = "PASS" if passed else "FAIL — INVESTIGATE"

        print(
            f"  {str(group):<{label_width}} "
            f"rate: {rate * 100:5.1f}%  ratio: {ratio:.3f}  {status}"
        )
        summary[group] = {"rate": rate, "ratio": ratio, "pass": passed}

    return summary

## Bias Analysis

### 01-Gender Bias

In [19]:
# Run the aggregation to get approval rates, average APR, and average approved amount by gender
pipeline_gender = [
    {
        "$group": {
            "_id": "$applicant_info.gender",
            "total": {"$sum": 1},
            "approved": {
                "$sum": {"$cond": ["$decision.loan_approved", 1, 0]}
            },
            "avg_interest_rate": {
                "$avg": {
                    "$cond": ["$decision.loan_approved", "$decision.interest_rate", None]
                }
            },
            "avg_approved_amount": {
                "$avg": {
                    "$cond": ["$decision.loan_approved", "$decision.approved_amount", None]
                }
            }
        }
    },
    {
        "$addFields": {
            "approval_rate": {"$divide": ["$approved", "$total"]}
        }
    },
    {"$sort": {"approval_rate": -1}}
]

gender_results = list(collection.aggregate(pipeline_gender))

print("\n  Approval Rates & Pricing by Gender:")
print(f"  {'Gender':<12} {'Total':>6} {'Approved':>9} "
      f"{'Rate':>8} {'Avg APR':>9} {'Avg Amount':>12}")
print("  " + "-" * 58)

for g in gender_results:
    apr = (f"{g['avg_interest_rate']:.2f}%" if g["avg_interest_rate"] else "N/A")
    amt = (f"${g['avg_approved_amount']:,.0f}" if g["avg_approved_amount"] else "N/A")
    print(
        f"  {str(g['_id']):<12} {g['total']:>6} {g['approved']:>9} "
        f"{g['approval_rate'] * 100:>7.1f}% {apr:>9} {amt:>12}"
    )


  Approval Rates & Pricing by Gender:
  Gender        Total  Approved     Rate   Avg APR   Avg Amount
  ----------------------------------------------------------
  Unknown           2         2   100.0%     4.25%      $31,500
  Male            247       163    66.0%     4.63%      $48,963
  Female          250       126    50.4%     4.49%      $46,651


In [21]:
gender_summary = disparate_impact_report(gender_results)


  Disparate Impact Analysis (80% Rule):

  Reference group (highest rate): Unknown → 100.0%
  80% threshold: 80.0%

  Unknown              rate: 100.0%  ratio: 1.000  PASS
  Male                 rate:  66.0%  ratio: 0.660  FAIL — INVESTIGATE
  Female               rate:  50.4%  ratio: 0.504  FAIL — INVESTIGATE


### 02-Age Bias

In [31]:
# Age is a protected characteristic under:
#   - Age Discrimination Act / ECOA
#   - EU AI Act Art. 10 (prohibited use of age as discriminatory proxy)
# Note: Minors (<18) are flagged as a data quality issue since they cannot
# hold credit contracts under the AI Act.

# Fetch only the fields needed for age bias analysis
REF_DATE = pd.Timestamp(f"{REF_YEAR}-12-31")
for doc in raw_docs:
    dob_raw = (doc.get("applicant_info") or {}).get("date_of_birth")
    try:
        dob = pd.to_datetime(dob_raw)
        age = (REF_DATE - dob).days // 365
    except (ValueError, TypeError):
        age = None

    age_band = get_age_band(age)

raw_docs = list(collection.find(
    {},
    {
        "applicant_info.date_of_birth": 1,
        "decision.loan_approved": 1,
        "decision.interest_rate": 1
    }
))

# Initialise one bucket per age band
age_buckets = {
    band: {"total": 0, "approved": 0, "interest_rates": []}
    for band in AGE_BUCKET_ORDER
}

minor_ids = []
for doc in raw_docs:
    dob_raw = (doc.get("applicant_info") or {}).get("date_of_birth")

    # Compute numeric age for minor detection (reuse helpers)
    dob_date = parse_dob(dob_raw)
    age = compute_age(dob_date, reference_date=date(REF_YEAR, 12, 31))

    age_band = get_age_band(dob_raw)
    age_buckets[age_band]["total"] += 1

    decision = doc.get("decision") or {}
    if decision.get("loan_approved", False):
        age_buckets[age_band]["approved"] += 1

    interest_rate = decision.get("interest_rate")
    if interest_rate is not None:
        age_buckets[age_band]["interest_rates"].append(interest_rate)

    # Flag minors using computed numeric age
    if age is not None and age < 18:
        minor_ids.append(str(doc.get("_id")))

# Flag minors - cannot hold credit contracts (AI Act Art. 10 data quality obligation)
if minor_ids:
    print(
        f"\nDATA QUALITY — Apparent minors (<18) detected: "
        f"{len(minor_ids)} record(s). IDs: {minor_ids}"
    )
    print("  Action: Verify DOB; minors cannot hold credit contracts.")

print(f"\n  {'Age Band':<12} {'Total':>6} {'Approved':>9} {'Rate':>8} {'Avg APR':>9}")
print("  " + "-" * 47)

age_results = []
for band in AGE_BUCKET_ORDER:
    b = age_buckets[band]
    if b["total"] == 0:
        continue
    rate = b["approved"] / b["total"]
    avg_apr = avg_or_none(b["interest_rates"])
    apr_str = f"{avg_apr:.2f}%" if avg_apr is not None else "N/A"
    print(
        f"  {band:<12} {b['total']:>6} {b['approved']:>9} "
        f"{rate * 100:>7.1f}% {apr_str:>9}"
    )
    age_results.append({
        "_id": band,
        "total": b["total"],
        "approved": b["approved"],
        "approval_rate": rate
    })


  Age Band      Total  Approved     Rate   Avg APR
  -----------------------------------------------
  18-24            12         5    41.7%     4.32%
  25-34           150        69    46.0%     4.43%
  35-44           176       115    65.3%     4.56%
  45-54            88        57    64.8%     4.68%
  55-64            56        35    62.5%     4.51%
  65+              13         7    53.8%     5.37%
  Unknown           4         3    75.0%     4.93%


In [32]:
age_summary = disparate_impact_report(age_results)


  Disparate Impact Analysis (80% Rule):

  Reference group (highest rate): Unknown → 75.0%
  80% threshold: 60.0%

  Unknown              rate:  75.0%  ratio: 1.000  PASS
  35-44                rate:  65.3%  ratio: 0.871  PASS
  45-54                rate:  64.8%  ratio: 0.864  PASS
  55-64                rate:  62.5%  ratio: 0.833  PASS
  65+                  rate:  53.8%  ratio: 0.718  FAIL — INVESTIGATE
  25-34                rate:  46.0%  ratio: 0.613  FAIL — INVESTIGATE
  18-24                rate:  41.7%  ratio: 0.556  FAIL — INVESTIGATE


### 03-Geographic Bias

In [33]:
# Zip-code-based lending bias is a modern form of redlining. The AI Act
# (Recital 44, Art. 10) prohibits AI systems from using proxies that
# reproduce or amplify historical discrimination. Geographic clustering
# in the NovaCred dataset (LA 90xxx, NYC 10xxx, Atlanta 30xxx) may
# proxy for race/ethnicity and must be audited.

# Map ZIP prefixes to human-readable region names.
ZIP_PREFIX_MAP = {
    ("90", "91"): "Los Angeles, CA",
    ("10",):      "New York City, NY",
    ("30",):      "Atlanta, GA",
}

def zip_to_region(zip_code):
    """Return the region name for a given ZIP code string, or 'Other'/'Unknown'."""
    if zip_code is None:
        return "Unknown"
    zip_str = str(zip_code).strip()
    for prefixes, region in ZIP_PREFIX_MAP.items():
        if zip_str.startswith(prefixes):
            return region
    return "Other"

# Aggregate per ZIP code in MongoDB, then roll up into regions in Python
pipeline_geo = [
    {
        "$group": {
            "_id": "$applicant_info.zip_code",
            "total": {"$sum": 1},
            "approved": {
                "$sum": {"$cond": ["$decision.loan_approved", 1, 0]}
            },
            "avg_interest_rate": {
                "$avg": {
                    "$cond": ["$decision.loan_approved", "$decision.interest_rate", None]
                }
            }
        }
    }
]

zip_results = list(collection.aggregate(pipeline_geo))

region_buckets = {}
for z in zip_results:
    region = zip_to_region(z["_id"])
    if region not in region_buckets:
        region_buckets[region] = {"total": 0, "approved": 0, "interest_rates": []}

    region_buckets[region]["total"] += z["total"]
    region_buckets[region]["approved"] += z["approved"]

    if z["avg_interest_rate"]:
        region_buckets[region]["interest_rates"].append(z["avg_interest_rate"])

print(f"\n  {'Region':<22} {'Total':>6} {'Approved':>9} {'Rate':>8} {'Avg APR':>9}")
print("  " + "-" * 57)

geo_results = []
for region, b in sorted(region_buckets.items(), key=lambda x: -x[1]["approved"] / x[1]["total"]):
    rate = b["approved"] / b["total"] if b["total"] > 0 else 0
    avg_apr = avg_or_none(b["interest_rates"])
    apr_str = f"{avg_apr:.2f}%" if avg_apr is not None else "N/A"

    print(f"  {region:<22} {b['total']:>6} {b['approved']:>9} {rate * 100:>7.1f}% {apr_str:>9}")

    geo_results.append({"_id": region, "total": b["total"], "approved": b["approved"], "approval_rate": rate})


  Region                  Total  Approved     Rate   Avg APR
  ---------------------------------------------------------
  Other                       1         1   100.0%     5.10%
  New York City, NY         251       162    64.5%     4.43%
  Atlanta, GA                18        10    55.6%     4.89%
  Los Angeles, CA           229       118    51.5%     4.40%


In [34]:
geo_summary = disparate_impact_report(geo_results, label_width=22)


  Disparate Impact Analysis (80% Rule):

  Reference group (highest rate): Other → 100.0%
  80% threshold: 80.0%

  Other                  rate: 100.0%  ratio: 1.000  PASS
  New York City, NY      rate:  64.5%  ratio: 0.645  FAIL — INVESTIGATE
  Atlanta, GA            rate:  55.6%  ratio: 0.556  FAIL — INVESTIGATE
  Los Angeles, CA        rate:  51.5%  ratio: 0.515  FAIL — INVESTIGATE


### 04-Spending Category Bias

In [35]:
# The dataset includes categories such as 'Gambling', 'Adult Entertainment',
# and 'Alcohol'. Using such categories as model inputs may constitute
# discrimination by proxy (AI Act Art. 10 §5, Recital 44)

SENSITIVE_CATEGORIES = ["Gambling", "Adult Entertainment", "Alcohol"]

pipeline_spending = [
    {"$unwind": "$spending_behavior"},
    {
        "$group": {
            "_id": "$spending_behavior.category",
            "total_applicants": {"$sum": 1},
            "approved": {
                "$sum": {
                    "$cond": ["$decision.loan_approved", 1, 0]
                }
            },
            "avg_amount": {
                "$avg": "$spending_behavior.amount"
            }
        }
    },
    {
        "$addFields": {
            "approval_rate": {
                "$divide": ["$approved", "$total_applicants"]
            }
        }
    },
    {"$sort": {"approval_rate": 1}}
]

spending_results = list(collection.aggregate(pipeline_spending))

# Print spending category table
print(f"\n  {'Category':<25} {'Applications':>12} {'Approved':>9} {'Rate':>8} {'Avg Spend':>11}")
print("  " + "-" * 67)

flagged = []

for cat in spending_results:
    flag = " SENSITIVE" if cat["_id"] in SENSITIVE_CATEGORIES else ""

    if cat["_id"] in SENSITIVE_CATEGORIES:
        flagged.append(cat)

    print(f"  {str(cat['_id']):<25} {cat['total_applicants']:>12} "
          f"{cat['approved']:>9} {cat['approval_rate'] * 100:>7.1f}% "
          f"${cat['avg_amount']:>9.0f}{flag}")

# Flag sensitive categories
if flagged:
    print("\n Sensitive spending categories detected "
          "as potential model inputs:")

    for cat in flagged:
        print(f"     • '{cat['_id']}': {cat['approval_rate'] * 100:.1f}% "
              f"approval rate across {cat['total_applicants']} applicants.")

    print("  Action: Verify that 'Gambling', 'Adult Entertainment', and "
          "'Alcohol' spending\n"
          "  are NOT used as model features. If so, remove per Art. 10 §5 "
          "(prohibited\n"
          "  data use) and re-train. Document in Technical Documentation "
          "(Art. 11).")


  Category                  Applications  Approved     Rate   Avg Spend
  -------------------------------------------------------------------
  Rent                                59        26    44.1% $      524
  Dining                              65        31    47.7% $      473
  Fitness                             69        35    50.7% $      452
  Entertainment                       72        41    56.9% $      496
  Gambling                             7         4    57.1% $      457 SENSITIVE
  Healthcare                          68        39    57.4% $      451
  Education                           64        37    57.8% $      500
  Groceries                           65        38    58.5% $      488
  Adult Entertainment                  5         3    60.0% $      591 SENSITIVE
  Travel                              80        48    60.0% $      493
  Transportation                      61        38    62.3% $      439
  Utilities                           76        49    64

### 05-Rejection Fairness

In [36]:
# The rejection reason 'algorithm_risk_score' is opaque and may mask bias.
# Under AI Act Art. 13 (Transparency) and Art. 14 (Human Oversight),
# applicants must receive meaningful explanations for adverse decisions.

pipeline_rejection = [
    {"$match": {"decision.loan_approved": False}},
    {
        "$group": {
            "_id": {
                "gender": "$applicant_info.gender",
                "reason": "$decision.rejection_reason"
            },
            "count": {"$sum": 1}
        }
    },
    {"$sort": {"_id.gender": 1, "count": -1}}
]

rejection_results = list(collection.aggregate(pipeline_rejection))

# Restructure results for display
gender_rejections = {}

for r in rejection_results:
    gender = r["_id"]["gender"]
    reason = r["_id"]["reason"] or "Unknown"

    if gender not in gender_rejections:
        gender_rejections[gender] = {}

    gender_rejections[gender][reason] = r["count"]

# Print rejection reason table
print("\n  Rejection reasons by gender:\n")

all_reasons = sorted({
    reason
    for g in gender_rejections.values()
    for reason in g
})

header = f"  {'Reason':<35}"
for g in sorted(gender_rejections.keys()):
    header += f" {g:>10}"
print(header)
print("  " + "-" * (35 + 11 * len(gender_rejections)))

for reason in all_reasons:
    row = f"  {reason:<35}"
    for g in sorted(gender_rejections.keys()):
        count = gender_rejections.get(g, {}).get(reason, 0)
        row += f" {count:>10}"
    print(row)

# Flag opaque algorithmic rejections
print("\n AI ACT ART. 13 CHECK — 'algorithm_risk_score' rejections:")

for gender, reasons in gender_rejections.items():
    total = sum(reasons.values())
    algo = reasons.get("algorithm_risk_score", 0)
    algo_pct = algo / total * 100 if total > 0 else 0
    flag = " HIGH" if algo_pct > 50 else ""

    print(f"    {gender}: {algo}/{total} = {algo_pct:.1f}% opaque algorithmic rejections{flag}")

print("\n  Action: If 'algorithm_risk_score' represents >50% of rejections")
print("  for any protected group, conduct root-cause analysis and")
print("  implement human review (Art. 14 Human Oversight obligation).")


  Rejection reasons by gender:

  Reason                                  Female       Male
  ---------------------------------------------------------
  algorithm_risk_score                       100         69
  high_dti_ratio                               8          4
  insufficient_credit_history                 15          8
  low_income                                   1          3

 AI ACT ART. 13 CHECK — 'algorithm_risk_score' rejections:
    Female: 100/124 = 80.6% opaque algorithmic rejections HIGH
    Male: 69/84 = 82.1% opaque algorithmic rejections HIGH

  Action: If 'algorithm_risk_score' represents >50% of rejections
  for any protected group, conduct root-cause analysis and
  implement human review (Art. 14 Human Oversight obligation).


### 06-Intersectional Bias: Gender and Age

In [37]:
# The AI Act (Recital 44) recognises that discrimination often operates
# at the intersection of protected characteristics. A system may appear
# fair on each dimension individually while still discriminating against
# specific subgroups (e.g., young women or older men).
# This section applies the 80% rule to gender × age intersections.


# Fetch only the fields needed for intersectional analysis
raw_docs_intersect = list(collection.find(
    {},
    {
        "applicant_info.gender": 1,
        "applicant_info.age": 1,
        "decision.loan_approved": 1
    }
))

# Build intersectional buckets (gender × age band)
intersect_buckets = {}
for doc in raw_docs_intersect:
    info = doc.get("applicant_info") or {}
    gender = info.get("gender", "Unknown")
    age_band = get_age_band(info.get("age"))
    key = f"{gender} / {age_band}"

    bucket = intersect_buckets.setdefault(key, {"total": 0, "approved": 0})
    bucket["total"] += 1
    if (doc.get("decision") or {}).get("loan_approved", False):
        bucket["approved"] += 1

print(f"\n  {'Subgroup':<25} {'Total':>6} {'Approved':>9} {'Rate':>8}")
print("  " + "-" * 50)

intersect_results = []
for key, b in sorted(intersect_buckets.items(), key=lambda x: -x[1]["approved"] / x[1]["total"] if x[1]["total"] > 0 else 0):
    if b["total"] < 5:  # Skip cells too small to be statistically reliable.
        continue

    rate = b["approved"] / b["total"]
    print(f"  {key:<25} {b['total']:>6} {b['approved']:>9} {rate * 100:>7.1f}%")

    intersect_results.append({"_id": key, "total": b["total"], "approved": b["approved"], "approval_rate": rate})



  Subgroup                   Total  Approved     Rate
  --------------------------------------------------
  Male / Unknown               247       163    66.0%
  Female / Unknown             250       126    50.4%


In [38]:
intersect_summary = disparate_impact_report(intersect_results, label_width=25)


  Disparate Impact Analysis (80% Rule):

  Reference group (highest rate): Male / Unknown → 66.0%
  80% threshold: 52.8%

  Male / Unknown            rate:  66.0%  ratio: 1.000  PASS
  Female / Unknown          rate:  50.4%  ratio: 0.764  FAIL — INVESTIGATE


### Compliance Summary

In [39]:
# EU AI Act Compliance Summary
EU_AI_ACT_ACTIONS = [
    "Art. 9  — Update risk management to cover bias risks   ",
    "Art. 10 — Audit training data for discriminatory labels",
    "Art. 11 — Document findings in Technical Documentation ",
    "Art. 13 — Provide meaningful rejection explanations    ",
    "Art. 14 — Implement human review for flagged decisions ",
    "Art. 29 — Notify deployer (NovaCred) of audit results  ",
]

all_analyses = [
    ("Gender (Art. 10)", gender_summary),
    ("Age (Art. 10)", age_summary),
    ("Geography / Redlining", geo_summary),
    ("Intersectional (Gender × Age)", intersect_summary),
]

any_fail = False

# Print compliance table
print(f"\n {'Analysis':<35} {'Status':<20} {'Failing Groups'}")
print("  " + "-" * 75)

for label, di_results in all_analyses:
    if not di_results:
        print(f"  {label:<35} {'NO DATA':<20}")
        continue

    failing = [g for g, v in di_results.items() if not v["pass"]]

    if failing:
        any_fail = True
        status = "FAIL"
        failing_str = ", ".join(str(f) for f in failing[:3])
    else:
        status = "PASS"
        failing_str = "—"

    print(f"  {label:<35} {status:<20} {failing_str}")

print()

# Overall verdict
if any_fail:
    print("OVERALL VERDICT: DISPARATE IMPACT DETECTED")
    print("\nRequired Actions under EU AI Act:")
    print("┌─────────────────────────────────────────────────────────┐")
    for action in EU_AI_ACT_ACTIONS:
        print(f"│ {action} │")
    print("└─────────────────────────────────────────────────────────┘")
else:
    print("OVERALL VERDICT: No statistically significant disparate impact detected.")
    print("Recommendation: Continue quarterly monitoring as required by Art. 9.")

print(f"Records audited : {collection.count_documents({})}")


 Analysis                            Status               Failing Groups
  ---------------------------------------------------------------------------
  Gender (Art. 10)                    FAIL                 Male, Female
  Age (Art. 10)                       FAIL                 65+, 25-34, 18-24
  Geography / Redlining               FAIL                 New York City, NY, Atlanta, GA, Los Angeles, CA
  Intersectional (Gender × Age)       FAIL                 Female / Unknown

OVERALL VERDICT: DISPARATE IMPACT DETECTED

Required Actions under EU AI Act:
┌─────────────────────────────────────────────────────────┐
│ Art. 9  — Update risk management to cover bias risks    │
│ Art. 10 — Audit training data for discriminatory labels │
│ Art. 11 — Document findings in Technical Documentation  │
│ Art. 13 — Provide meaningful rejection explanations     │
│ Art. 14 — Implement human review for flagged decisions  │
│ Art. 29 — Notify deployer (NovaCred) of audit results   │
└────────────────